In [1]:
# analyse size and time of the estimated timetable
import os, json, time
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
URL = "https://prim.iledefrance-mobilites.fr/marketplace/estimated-timetable"

start = time.time()
r = requests.get(URL, headers={"apikey": os.getenv("PRIM_API_KEY")},
                 params={"LineRef": "ALL"}, timeout=90)
print(r.status_code, f"{len(r.content) / 1e6:.1f} MB in {time.time() - start:.1f} s")
data = r.json()

200 25.8 MB in 1.3 s


In [2]:
# save one raw response 
with open("../data/raw/sample_estimated_timetable.json", "w") as f:
    json.dump(data, f)

# look at structure of response
print(data.keys())
print(data["Siri"]["ServiceDelivery"].keys())

dict_keys(['Siri'])
dict_keys(['ResponseTimestamp', 'ProducerRef', 'ResponseMessageIdentifier', 'EstimatedTimetableDelivery'])


In [ ]:
# count trains per line, and check all our lines are here
from collections import Counter

# keep only the data we need (remove useless parent elements)
journeys = [
    j
    for d in data["Siri"]["ServiceDelivery"]["EstimatedTimetableDelivery"]
    for frame in d["EstimatedJourneyVersionFrame"]
    for j in frame["EstimatedVehicleJourney"]
]

lines = Counter(j["LineRef"]["value"] for j in journeys)
# count total number of trains
print(len(journeys), "trains in the feed")

import sys
sys.path.insert(0, "..")            
from src.config import FOCUS_LINES
# count number of trains per line
for name, ref in FOCUS_LINES.items():
    print(f"{name:<9} {ref:<22} {lines.get(ref, 0)} trains")

10941 trains in the feed
RER A     STIF:Line::C01742:     69 trains
RER B     STIF:Line::C01743:     42 trains
Metro 13  STIF:Line::C01383:     77 trains
Metro 14  STIF:Line::C01384:     25 trains
RER C     STIF:Line::C01727:     12 trains
Transilien N STIF:Line::C01736:     7 trains


In [4]:
# looking at one train
rer_a = next(j for j in journeys if j["LineRef"]["value"] == FOCUS_LINES["RER A"])
print(json.dumps(rer_a, indent=2)[:3000])

{
  "RecordedAtTime": "2026-11-05T19:49:59.667Z",
  "LineRef": {
    "value": "STIF:Line::C01742:"
  },
  "DirectionRef": {
    "value": "Retour"
  },
  "DatedVehicleJourneyRef": {
    "value": "SNCF_MAGENTA_PRD:VehicleJourney::bc37f9f6-5178-42c9-833e-ffdc6a84676c:LOC"
  },
  "VehicleMode": [
    "RAIL"
  ],
  "VehicleFeatureRef": [
    "longTrain"
  ],
  "RouteRef": {},
  "PublishedLineName": [
    {
      "value": "A"
    }
  ],
  "DirectionName": [],
  "OriginRef": {
    "value": "STIF:StopArea:SP:43198:"
  },
  "OriginName": [
    {
      "value": "Saint-Germain-en-Laye"
    }
  ],
  "DestinationRef": {
    "value": "STIF:StopArea:SP:43239:"
  },
  "DestinationName": [
    {
      "value": "Marne-la-Vall\u00e9e Chessy"
    }
  ],
  "OperatorRef": {},
  "ProductCategoryRef": {},
  "JourneyNote": [
    {
      "value": "QIKI"
    }
  ],
  "EstimatedCalls": {
    "EstimatedCall": [
      {
        "StopPointRef": {
          "value": "STIF:StopArea:SP:43224:"
        },
        "Expec